# Clase 8 — Clasificación: comparar algoritmos y elegir el mejor

En la clase 7 aprendimos a construir un flujo reproducible con `Pipeline`. Ahora damos el siguiente paso: en vez de trabajar con un único algoritmo, vamos a entrenar tres y elegir el mejor con criterio, no por intuición.

En la clase 6 conocimos las familias de algoritmos de forma conceptual. Esta es la **primera vez que los ponemos en práctica** sobre datos reales: KNN, árbol de decisión y regresión logística compiten en el mismo experimento y con el mismo split.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Los tres algoritmos: de la teoría a la práctica |
| 2 | Preparar el experimento |
| 3 | Entrenar y comparar |
| 4 | Matriz de confusión |
| 5 | Precision vs. Recall: cuándo importa cuál |
| 6 | Elegir el modelo final con criterio |
| 7 | Actividad: selección de modelo con justificación |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

print("Librerías listas.")

---
## 1. Los tres algoritmos: de la teoría a la práctica

En la clase 6 vimos que los algoritmos de clasificación se pueden agrupar según su forma de aprender. Hoy los vemos en código por primera vez sobre un dataset real.

| Algoritmo | Idea central | Fortaleza | Debilidad |
|---|---|---|---|
| **KNN** | Clasificar según los vecinos más cercanos en el espacio de features | Simple, sin supuestos, captura no-linearidades | Lento con muchos datos, muy sensible a la escala |
| **Decision Tree** | Dividir el espacio con reglas if/else aprendidas de los datos | Interpretable, no requiere escalar | Tiende a sobreajustar si no se limita la profundidad |
| **Logistic Regression** | Estimar la probabilidad de cada clase con una función lineal | Rápido, probabilidades calibradas, coeficientes interpretables | Solo captura relaciones lineales |

> 💡 Ninguno es universalmente mejor. El modelo correcto depende del dataset, del contexto y de qué tipo de error tiene mayor costo. Lo que vemos hoy es exactamente cómo tomar esa decisión con evidencia.

---
## 2. Preparar el experimento

Usamos el dataset Breast Cancer: 569 muestras, 30 features, 2 clases (maligno/benigno). Para comparar algoritmos con justicia, todos reciben exactamente el mismo train/test split y los mismos datos escalados.

In [ ]:
# ─── Cargar y dividir ─────────────────────────────────────────────────────────
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
clases = cancer.target_names   # ['malignant', 'benign']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar: necesario para KNN y Logistic Regression, no impacta a Decision Tree
# Escalamos con los parámetros del train (fit solo sobre train)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # aprende media/std del train
X_test_sc  = scaler.transform(X_test)        # aplica la misma transformación

print(f"Train: {X_train.shape[0]} — Test: {X_test.shape[0]}")
print(f"Clases: {clases}")
print(f"Balance test: {dict(zip(*np.unique(y_test, return_counts=True)))}")

---
## 3. Entrenar y comparar

In [ ]:
# ─── Definir los tres modelos ─────────────────────────────────────────────────
modelos = {
    "KNN (k=5)":              KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (d=4)":   DecisionTreeClassifier(max_depth=4, random_state=42),
    "Logistic Regression":   LogisticRegression(max_iter=1000, random_state=42),
}

# ─── Entrenar todos y guardar métricas ────────────────────────────────────────
resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_sc, y_train)       # entrenar
    y_pred = modelo.predict(X_test_sc)    # predecir

    resultados.append({
        "modelo":    nombre,
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),   # para clase 1 (benign)
        "recall":    recall_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred),
    })

df_resultados = pd.DataFrame(resultados).set_index("modelo")
df_resultados.round(4)

In [ ]:
# ─── Visualizar comparación ───────────────────────────────────────────────────
metricas = ["accuracy", "precision", "recall", "f1"]

fig, axes = plt.subplots(1, len(metricas), figsize=(14, 4))

for ax, metrica in zip(axes, metricas):
    valores = df_resultados[metrica]
    colores = ["steelblue", "coral", "seagreen"]
    bars = ax.bar(valores.index, valores.values, color=colores)
    ax.set_title(metrica.capitalize())
    ax.set_ylim(0.85, 1.0)   # zoom en zona relevante
    ax.set_xticklabels(valores.index, rotation=20, ha="right", fontsize=8)
    # Anotar el valor en cada barra
    for bar, val in zip(bars, valores.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.3f}", ha="center", fontsize=8)

plt.suptitle("Comparación de modelos — Breast Cancer", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Matriz de confusión

La accuracy sola no cuenta toda la historia. La **matriz de confusión** muestra exactamente qué tipo de errores comete cada modelo.

In [ ]:
# ─── Matrices de confusión para los tres modelos ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (nombre, modelo) in zip(axes, modelos.items()):
    y_pred = modelo.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Pred Maligno", "Pred Benigno"],
        yticklabels=["Real Maligno", "Real Benigno"],
        ax=ax
    )
    ax.set_title(nombre, fontsize=10)

plt.suptitle("Matrices de confusión", fontsize=13)
plt.tight_layout()
plt.show()

| Celda de la matriz | Nombre | Significado |
|---|---|---|
| Arriba-izquierda | **Verdadero Negativo (TN)** | Predijo maligno y era maligno |
| Arriba-derecha | **Falso Positivo (FP)** | Predijo benigno pero era maligno |
| Abajo-izquierda | **Falso Negativo (FN)** | Predijo maligno pero era benigno |
| Abajo-derecha | **Verdadero Positivo (TP)** | Predijo benigno y era benigno |

---
## 5. Precision vs. Recall: cuándo importa cuál

Para elegir el modelo correcto hay que entender el contexto del problema. Estas dos métricas miden cosas distintas y a veces se contradicen.

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{— de los que dije positivos, ¿cuántos lo eran?}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{— de todos los positivos reales, ¿cuántos detecté?}$$

| Contexto | Qué error es más costoso | Métrica a priorizar |
|---|---|---|
| Diagnóstico médico (detectar cáncer) | Falso Negativo (no detectar un tumor real) | **Recall** |
| Filtro de spam | Falso Positivo (eliminar email legítimo) | **Precision** |
| Fraude bancario | Falso Negativo (no detectar fraude) | **Recall** |
| Recomendación de producto | Falso Positivo (recomendar algo irrelevante) | **Precision** |

In [ ]:
# ─── Análisis de errores en contexto médico ───────────────────────────────────
# Para diagnóstico de cáncer, nos importa más el Recall de maligno:
# queremos detectar todos los tumores malignos posibles.

print("Reporte detallado — foco en clase 'malignant':")
print()

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)
    fn = cm[0, 1]   # Falsos negativos: malignos clasificados como benignos
    print(f"{nombre}: {fn} tumores malignos clasificados como benignos (falsos negativos)")

---
## 6. Elegir el modelo final con criterio

In [ ]:
# ─── Tabla resumen para la decisión ──────────────────────────────────────────
print("Tabla comparativa completa:")
print(df_resultados.round(4).to_string())
print()

# ─── Mejor modelo por F1 (balance precision/recall) ──────────────────────────
mejor = df_resultados["f1"].idxmax()
print(f"Mejor F1: {mejor} ({df_resultados.loc[mejor, 'f1']:.4f})")
print()

# ─── Reporte detallado del mejor modelo ───────────────────────────────────────
y_pred_mejor = modelos[mejor].predict(X_test_sc)
print(f"Classification report — {mejor}:")
print(classification_report(y_test, y_pred_mejor, target_names=clases))

---
## 7. Actividad: selección de modelo con justificación

Repetís el experimento completo, pero esta vez el foco no está en el código sino en la **toma de decisión**: elegir un modelo según el contexto, la métrica relevante y el costo del error.

El código está casi completo. Tu trabajo es ejecutarlo, analizar los resultados y completar la justificación escrita al final.

In [ ]:
# ─── Dataset: wine (clasificación de vinos en 3 variedades) ──────────────────
from sklearn.datasets import load_wine

wine = load_wine()
X_w, y_w = wine.data, wine.target

print(f"Dataset: {X_w.shape[0]} muestras, {X_w.shape[1]} features, {len(wine.target_names)} clases")
print(f"Clases: {wine.target_names}")
print()
print("Contexto: sos parte del equipo de control de calidad de una bodega.")
print("El modelo debe identificar de qué variedad de uva proviene cada lote.")
print("Un error en la clasificación puede afectar el etiquetado y la comercialización.")

In [ ]:
# ─── Experimento completo con el dataset wine ─────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_w, y_w, test_size=0.2, random_state=42, stratify=y_w
)

# Escalar (los parámetros se aprenden solo del train)
scaler_w = StandardScaler()
X_train_ws = scaler_w.fit_transform(X_train_w)
X_test_ws  = scaler_w.transform(X_test_w)

# Entrenar los tres modelos
modelos_w = {
    "KNN (k=5)":            KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (d=4)":  DecisionTreeClassifier(max_depth=4, random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
}

resultados_w = []
for nombre, modelo in modelos_w.items():
    modelo.fit(X_train_ws, y_train_w)
    y_pred_w = modelo.predict(X_test_ws)
    resultados_w.append({
        "modelo":           nombre,
        "accuracy":         accuracy_score(y_test_w, y_pred_w),
        "f1_macro":         f1_score(y_test_w, y_pred_w, average="macro"),
        "precision_macro":  precision_score(y_test_w, y_pred_w, average="macro"),
        "recall_macro":     recall_score(y_test_w, y_pred_w, average="macro"),
    })

df_resultados_w = pd.DataFrame(resultados_w).set_index("modelo")
print("Tabla comparativa — Dataset Wine:")
df_resultados_w.round(4)

In [ ]:
# ─── Justificación de modelo elegido ──────────────────────────────────────────
# Completá este template con los resultados reales de tu experimento.
# No hay una única respuesta correcta: el objetivo es razonar con evidencia.

justificacion = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECCIÓN DE MODELO — Dataset Wine
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Modelo elegido: ...

2. Métricas del modelo elegido:
   - Accuracy:        ...
   - F1 macro:        ...
   - Precision macro: ...
   - Recall macro:    ...

3. Contexto del problema:
   - ¿Quién usa estas predicciones?: ...
   - ¿Qué consecuencia tiene una clasificación incorrecta?: ...

4. Costo del error:
   - ¿Qué significa un error de clasificación en este contexto?: ...
   - ¿Hay alguna variedad donde equivocarse sea más costoso? ¿Por qué?: ...

5. Por qué este modelo sobre los otros:
   ...

6. Limitaciones del modelo elegido:
   ...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(justificacion)

---
## Entregable

Guardá el notebook con las celdas ejecutadas.
El entregable es el experimento con `wine`: tabla comparativa de los 3 modelos + justificación escrita del modelo elegido.

**Para la próxima clase:** regresión y documentación del baseline para el Proyecto de Módulo.